# Dictionary Discovery Workflow v3 - Complete Implementation

## Overview
Systematic dictionary-based topic discovery and model training workflow with structured file organization.

### Checkpoints:
- **CHECKPOINT 0**: Initial Setup → Create folders, load config
- **CHECKPOINT 1**: Text Processing → Chunk corpus
- **CHECKPOINT 2**: Vocabulary Building → Build vocab from chunks
- **CHECKPOINT 3**: Dictionary Expansion → Expand keywords (⚠️ MANUAL CURATION REQUIRED)
- **CHECKPOINT 4**: Topic Vectors → Build weighted topic vectors
- **CHECKPOINT 5**: Chunk Scoring → Score & classify by confidence
- **CHECKPOINT 6**: Training Data Prep → Create train/val splits
- **CHECKPOINT 7**: Model Training → Train BERTJE
- **CHECKPOINT 8**: Visualizations → Generate plots

### Folder Structure:
```
workflow_data/{ModelType}-{Topic}_{Date}_{Version}/
  ├── config/
  ├── Dictionary/
  │   └── Dictionary_suggestions/
  ├── Model_finetuning/
  ├── Cosine_labeling/
  ├── Bertje_labeling/
  ├── Visuals/
  └── Other_data/
```

---
# CHECKPOINT 0: Initial Setup
---

In [ ]:
# ============================================================
# CELL 0.1: IMPORTS
# ============================================================
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

# NLTK
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# ML libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer

# Suppress warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("✓ All imports successful")

In [ ]:
# ============================================================
# CELL 0.2: CONFIGURATION
# ============================================================

CONFIG = {
    # =====================
    # WORKFLOW METADATA
    # =====================
    "workflow": {
        # Model type: "Pretrained" or "Finetuned_{source}"
        "model_type": "Pretrained",
        
        # Topic: "Slavery", "Policy", "Slavery-Policy", etc.
        "topic": "Slavery",
        
        # Version (None for auto-increment)
        "version": None,
    },
    
    # =====================
    # PATHS
    # =====================
    "paths": {
        "corpus_dir": "/home/user/policy-analysis/Slavery_text",
        "dictionary_excel": "/home/user/policy-analysis/dutch_slavery_legacy_dictionary.xlsx",
        "workflow_base": "/home/user/policy-analysis/workflow_data",
        "pretrained_model_path": None,
    },
    
    # =====================
    # MODEL SETTINGS
    # =====================
    "model": {
        "base_model_name": "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers",
        "use_pretrained": False,
    },
    
    # =====================
    # DICTIONARY SETTINGS
    # =====================
    "dictionary": {
        "use_excel": True,
        "topic_column": "topic",
        "keyword_column": "keyword",
        "sheet_name": 0,
        "default_topics": {
            "Historical slavery": ["slavernij", "tot-slaaf-gemaakte", "dwangarbeid", "zweep"],
            "Colonialism": ["kolonie", "koloniaal", "voc", "wic", "exploitatie"],
            "Modern racism& inequality": ["racisme", "discriminatie", "ongelijkheid"],
        },
    },
    
    # =====================
    # TEXT PROCESSING
    # =====================
    "chunking": {
        "sentences_per_chunk": 10,
        "min_sentences_to_keep": 3,
        "drop_likely_english": True,
        "remove_stopwords": True,
        "use_stemming": False,
    },
    
    "tokenize": {
        "lower": True,
        "keep_hyphen": True,
        "min_len": 2,
        "max_len": 30,
        "pattern": r"[0-9A-Za-zÀ-ÖØ-öø-ÿ\-]+",
    },
    
    # =====================
    # VOCABULARY SETTINGS
    # =====================
    "vocab": {
        "min_df": 5,
        "max_vocab": 50000,
    },
    
    # =====================
    # EXPANSION SETTINGS
    # =====================
    "expand": {
        "k_nearest": 50,
        "topN_per_topic": 300,
        "min_cosine": 0.55,
    },
    
    # =====================
    # SCORING SETTINGS
    # =====================
    "scoring": {
        "use_sif": True,
        "sif_a": 1e-3,
        "high_confidence_score": 0.50,
        "high_confidence_margin": 0.05,
        "low_confidence_score": 0.40,
        "low_confidence_margin": 0.02,
    },
    
    # =====================
    # TRAINING SETTINGS
    # =====================
    "training": {
        "num_epochs": 3,
        "batch_size_train": 16,
        "batch_size_eval": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "dataset_option": "option4",
    },
}

print("✓ Configuration loaded")
print(f"\nWorkflow: {CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}")

In [ ]:
# ============================================================
# CELL 0.3: FILE SYSTEM UTILITIES
# ============================================================

class WorkflowFileSystem:
    """Manages structured folder system for workflow data."""
    
    def __init__(self, config):
        self.config = config
        self.root = None
        self.folders = {}
    
    def create_workflow_folder(self):
        """Create main workflow folder with subfolders."""
        model_type = self.config["workflow"]["model_type"]
        topic = self.config["workflow"]["topic"]
        date = datetime.now().strftime("%m.%d.%y")
        
        version = self.config["workflow"]["version"]
        if version is None:
            version = self._get_next_version(model_type, topic, date)
        
        folder_name = f"{model_type}-{topic}_{date}_{version}"
        base_dir = self.config["paths"]["workflow_base"]
        self.root = Path(base_dir) / folder_name
        self.root.mkdir(parents=True, exist_ok=True)
        
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary/Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Other_data",
        ]
        
        for subfolder in subfolder_names:
            path = self.root / subfolder
            path.mkdir(parents=True, exist_ok=True)
            key = subfolder.split("/")[-1]
            self.folders[key] = path
        
        self.folders["Dictionary"] = self.root / "Dictionary"
        
        print(f"\n{'='*60}")
        print("WORKFLOW FOLDER CREATED")
        print(f"{'='*60}")
        print(f"Location: {self.root}")
        print(f"\nSubfolders:")
        for name in subfolder_names:
            print(f"  ✓ {name}/")
        
        return self.root
    
    def _get_next_version(self, model_type, topic, date):
        """Auto-increment version number."""
        base_dir = Path(self.config["paths"]["workflow_base"])
        if not base_dir.exists():
            return "v1"
        
        prefix = f"{model_type}-{topic}_{date}_v"
        existing = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith(prefix)]
        
        if not existing:
            return "v1"
        
        versions = []
        for folder in existing:
            try:
                version_str = folder.split("_v")[-1]
                versions.append(int(version_str.replace("v", "")))
            except:
                continue
        
        if versions:
            return f"v{max(versions) + 1}"
        return "v1"
    
    def load_existing_workflow(self, folder_path):
        """Load existing workflow folder."""
        self.root = Path(folder_path)
        if not self.root.exists():
            raise ValueError(f"Workflow folder not found: {folder_path}")
        
        subfolder_names = [
            "config", "Dictionary", "Dictionary_suggestions",
            "Model_finetuning", "Cosine_labeling", "Bertje_labeling",
            "Visuals", "Other_data"
        ]
        
        for name in subfolder_names:
            if name == "Dictionary_suggestions":
                path = self.root / "Dictionary" / name
            else:
                path = self.root / name
            if path.exists():
                self.folders[name] = path
        
        print(f"✓ Loaded existing workflow: {self.root.name}")
        return self.root
    
    def save_config(self, checkpoint_name=None):
        """Save CONFIG to config folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"config_{checkpoint_name}_{timestamp}.json" if checkpoint_name else f"config_{timestamp}.json"
        config_path = self.folders["config"] / filename
        
        config_data = {
            "metadata": {
                "timestamp": timestamp,
                "checkpoint": checkpoint_name,
                "workflow_folder": str(self.root),
            },
            "config": self.config
        }
        
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Config saved: {config_path.name}")
        return config_path
    
    def save_data(self, data, filename, folder_key, file_format="csv"):
        """Save data to specific folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        full_filename = f"{filename}.{file_format}"
        filepath = folder / full_filename
        
        if file_format == "csv":
            if not isinstance(data, pd.DataFrame):
                raise ValueError("CSV format requires DataFrame")
            data.to_csv(filepath, index=False, encoding='utf-8')
        elif file_format == "json":
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        elif file_format == "npy":
            np.save(filepath, data, allow_pickle=True)
        else:
            raise ValueError(f"Unsupported format: {file_format}")
        
        print(f"✓ Saved: {folder_key}/{full_filename}")
        return filepath
    
    def copy_file_to_folder(self, source_path, folder_key, new_name=None):
        """Copy external file to workflow folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        source = Path(source_path)
        if not source.exists():
            raise FileNotFoundError(f"Source file not found: {source_path}")
        
        dest_name = new_name if new_name else source.name
        dest_path = folder / dest_name
        
        shutil.copy2(source, dest_path)
        print(f"✓ Copied: {source.name} → {folder_key}/{dest_name}")
        return dest_path

print("✓ WorkflowFileSystem class defined")

In [ ]:
# ============================================================
# CELL 0.4: CREATE OR LOAD WORKFLOW
# ============================================================

# Choose one:
CREATE_NEW = True  # Set to False to load existing
EXISTING_FOLDER = None  # Set path if loading existing

fs = WorkflowFileSystem(CONFIG)

if CREATE_NEW:
    workflow_root = fs.create_workflow_folder()
    fs.save_config("initial_setup")
else:
    if EXISTING_FOLDER is None:
        raise ValueError("EXISTING_FOLDER must be set when CREATE_NEW=False")
    workflow_root = fs.load_existing_workflow(EXISTING_FOLDER)

print(f"\n✓ Workflow initialized: {workflow_root}")

In [ ]:
# ============================================================
# CELL 0.5: LOAD DICTIONARY
# ============================================================

def load_dictionary_from_excel(excel_path, config):
    """Load topics and keywords from Excel."""
    if not Path(excel_path).exists():
        print(f"⚠ Excel not found: {excel_path}")
        return config["dictionary"]["default_topics"]
    
    try:
        df = pd.read_excel(excel_path, sheet_name=config["dictionary"]["sheet_name"])
        topic_col = config["dictionary"]["topic_column"]
        keyword_col = config["dictionary"]["keyword_column"]
        
        if topic_col not in df.columns or keyword_col not in df.columns:
            return config["dictionary"]["default_topics"]
        
        topics_dict = {}
        for topic, group in df.groupby(topic_col):
            keywords = group[keyword_col].dropna().str.strip().tolist()
            if keywords:
                topics_dict[topic] = keywords
        
        print(f"✓ Loaded from Excel: {len(topics_dict)} topics, {len(df)} keywords")
        return topics_dict
    except Exception as e:
        print(f"⚠ Error: {e}")
        return config["dictionary"]["default_topics"]

if CONFIG["dictionary"]["use_excel"]:
    topics = load_dictionary_from_excel(CONFIG["paths"]["dictionary_excel"], CONFIG)
    CONFIG["topics"] = topics
    if Path(CONFIG["paths"]["dictionary_excel"]).exists():
        fs.copy_file_to_folder(
            CONFIG["paths"]["dictionary_excel"],
            "Dictionary",
            "input_dictionary.xlsx"
        )
else:
    CONFIG["topics"] = CONFIG["dictionary"]["default_topics"]

print(f"\n{'='*60}")
print("TOPICS LOADED")
print(f"{'='*60}")
for topic, keywords in CONFIG["topics"].items():
    print(f"  {topic}: {len(keywords)} keywords")

fs.save_config("with_dictionary")

✅ **CHECKPOINT 0 COMPLETE** - Folder structure created, dictionary loaded

---
# CHECKPOINT 1: Text Processing
---

Chunks corpus into sentence-based segments with cleaning.

In [ ]:
# ============================================================
# CELL 1.1: TEXT CLEANING UTILITIES
# ============================================================

stemmer = SnowballStemmer("dutch")

nltk_stopwords = set(stopwords.words('dutch')) | set(stopwords.words('english'))
custom_stopwords = set([
    "de","het","een","en","van","in","op","met","voor","tegen","zonder","bij",
    "naar","tot","uit","door","aan","om","te","als","ook","maar","want","dus",
    "of","dan","nog","wel","zijn","is","was","waren","worden","hebben","heeft",
    "had","doet","doen","al","alle","meer","minder","veel","weinig","binnen",
    "buiten","tussen","onder","boven","over","na","achter","naast","sinds",
    "tijdens","zoals","ik","jij","hij","zij","wij","jullie","u","je","ze",
    "dit","dat","die","deze","welke","ons","hun","hem","haar","bijlage",
    "bijlagen","inleiding","samenvatting","conclusie","conclusies","jaar","jaren",
])
ALL_STOPWORDS = nltk_stopwords | custom_stopwords

ENGLISH_HINTS = set("the and of to in that is for on with as by from at it this be are were was has have will would can could should".split())
DUTCH_HINTS = set("de het een en van voor met op aan te is zijn worden was waren niet bij in over uit door naar tot als ook om".split())

def likely_english_sentence(s: str) -> bool:
    if not CONFIG["chunking"]["drop_likely_english"]:
        return False
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", s.lower())
    if not tokens:
        return False
    e = sum(t in ENGLISH_HINTS for t in tokens)
    d = sum(t in DUTCH_HINTS for t in tokens)
    return e > max(2, d + 1)

def remove_stopwords_and_numbers(text: str) -> str:
    if pd.isna(text):
        return ""
    tokens = re.findall(r"\b\w+\b", text.lower())
    filtered = [tok for tok in tokens if tok not in ALL_STOPWORDS and not tok.isdigit()]
    return " ".join(filtered)

def stem_text(text: str) -> str:
    tokens = re.findall(r"\b\w+\b", text.lower())
    return " ".join(stemmer.stem(w) for w in tokens)

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def short_file_hash(path: str, n=8) -> str:
    return hashlib.sha1(path.encode("utf-8", errors="ignore")).hexdigest()[:n]

def make_chunk_uid(file_path: str, chunk_idx: int) -> str:
    return f"{short_file_hash(file_path)}:{chunk_idx:05d}"

print("✓ Text cleaning utilities ready")

In [ ]:
# ============================================================
# CELL 1.2: CHUNKING & PROCESSING
# ============================================================

def split_into_sentences(text: str) -> list:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def chunk_by_sentences(text: str, file_path: str) -> list:
    sentences = split_into_sentences(text)
    sentences_per_chunk = CONFIG["chunking"]["sentences_per_chunk"]
    min_sentences = CONFIG["chunking"]["min_sentences_to_keep"]
    
    chunks = []
    chunk_idx = 0
    
    for i in range(0, len(sentences), sentences_per_chunk):
        sent_slice = sentences[i:i + sentences_per_chunk]
        if len(sent_slice) < min_sentences:
            continue
        
        raw_text = " ".join(sent_slice)
        
        if CONFIG["chunking"]["drop_likely_english"]:
            filtered_sents = [s for s in sent_slice if not likely_english_sentence(s)]
            if len(filtered_sents) < min_sentences:
                text_for_scoring = ""
            else:
                text_for_scoring = normalize_space(" ".join(filtered_sents))
        else:
            text_for_scoring = normalize_space(raw_text)
        
        if CONFIG["chunking"]["remove_stopwords"] and text_for_scoring:
            text_for_scoring = remove_stopwords_and_numbers(text_for_scoring)
        
        if CONFIG["chunking"]["use_stemming"] and text_for_scoring:
            text_for_scoring = stem_text(text_for_scoring)
        
        text_for_scoring = normalize_space(text_for_scoring)
        
        chunks.append((
            make_chunk_uid(file_path, chunk_idx),
            raw_text,
            text_for_scoring,
            len(sent_slice)
        ))
        chunk_idx += 1
    
    return chunks

# Process corpus
print(f"\n{'='*60}")
print("PROCESSING CORPUS")
print(f"{'='*60}")

all_chunks = []
corpus_dir = Path(CONFIG["paths"]["corpus_dir"])

if not corpus_dir.exists():
    print(f"⚠ Corpus directory not found: {corpus_dir}")
else:
    doc_files = list(corpus_dir.glob("*.txt"))
    print(f"\nFound {len(doc_files)} documents")
    
    for doc_path in tqdm(doc_files, desc="Chunking documents"):
        with open(doc_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        
        doc_chunks = chunk_by_sentences(text, str(doc_path))
        for chunk_uid, raw_text, text_for_scoring, sentence_count in doc_chunks:
            all_chunks.append({
                'file_path': str(doc_path),
                'chunk_uid': chunk_uid,
                'raw_text': raw_text,
                'text_for_scoring': text_for_scoring,
                'sentence_count': sentence_count
            })
    
    chunks_df = pd.DataFrame(all_chunks)
    fs.save_data(chunks_df, "chunked_corpus", "Other_data", "csv")
    
    print(f"\n✓ Processed {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    print(f"  Avg sentences/chunk: {chunks_df['sentence_count'].mean():.1f}")
    print(f"  Empty scoring text: {(chunks_df['text_for_scoring'] == '').sum()}")
    
    fs.save_config("checkpoint1_chunks")

✅ **CHECKPOINT 1 COMPLETE** - Corpus chunked and saved

**Resume**: Load `chunks_df` from `Other_data/chunked_corpus.csv`

---
# CHECKPOINT 2: Vocabulary Building
---

Build vocabulary from corpus with frequency filtering.

In [ ]:
# ============================================================
# CELL 2.1: TOKENIZATION FOR VOCAB BUILDING
# ============================================================

_tok_re = re.compile(CONFIG["tokenize"]["pattern"])

def tokenize(text: str) -> list:
    if CONFIG["tokenize"]["lower"]:
        text = text.lower()
    toks = _tok_re.findall(text)
    keep = []
    mn = CONFIG["tokenize"]["min_len"]
    mx = CONFIG["tokenize"]["max_len"]
    for t in toks:
        if not CONFIG["tokenize"]["keep_hyphen"]:
            t = t.replace("-", "")
        if mn <= len(t) <= mx:
            keep.append(t)
    return keep

def read_text(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return path.read_text(errors="ignore")

print("✓ Tokenizer ready")

In [ ]:
# ============================================================
# CELL 2.2: BUILD VOCABULARY
# ============================================================

print(f"\n{'='*60}")
print("BUILDING VOCABULARY")
print(f"{'='*60}")

doc_tokens = []
term_freq = Counter()
doc_freq = Counter()

corpus_dir = Path(CONFIG["paths"]["corpus_dir"])
doc_files = list(corpus_dir.glob("*.txt"))

for fp in tqdm(doc_files, desc="Processing documents"):
    text = read_text(fp)
    toks = tokenize(text)
    doc_tokens.append((fp, toks))
    term_freq.update(toks)
    doc_freq.update(set(toks))

print(f"\n✓ Processed {len(doc_tokens)} documents")
print(f"  Total tokens: {sum(term_freq.values()):,}")
print(f"  Unique terms: {len(term_freq):,}")

# Filter vocabulary
min_df = CONFIG["vocab"]["min_df"]
max_vocab = CONFIG["vocab"]["max_vocab"]

vocab_candidates = [
    (term, freq) for term, freq in term_freq.items()
    if doc_freq[term] >= min_df
]

vocab_candidates.sort(key=lambda x: x[1], reverse=True)
vocab_candidates = vocab_candidates[:max_vocab]
terms = [term for term, _ in vocab_candidates]

print(f"\n✓ Filtered vocabulary: {len(terms)} terms")
print(f"  Min document frequency: {min_df}")
print(f"  Max vocabulary size: {max_vocab}")

# Save vocabulary
vocab_df = pd.DataFrame({
    'term': terms,
    'term_freq': [term_freq[t] for t in terms],
    'doc_freq': [doc_freq[t] for t in terms]
})
fs.save_data(vocab_df, "vocabulary", "Other_data", "csv")

# Save frequencies
freq_data = {
    'term_freq': dict(term_freq),
    'doc_freq': dict(doc_freq),
    'n_documents': len(doc_tokens)
}
fs.save_data(freq_data, "term_frequencies", "Other_data", "json")

fs.save_config("checkpoint2_vocab")

✅ **CHECKPOINT 2 COMPLETE** - Vocabulary built and saved

**Resume**: Load `terms`, `term_freq`, `doc_freq` from saved files

---
# CHECKPOINT 3: Dictionary Expansion
---

⚠️ **MANUAL CURATION REQUIRED AFTER THIS STEP**

Expand seed keywords using semantic similarity.

In [ ]:
# ============================================================
# CELL 3.1: LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print(f"\n{'='*60}")
print("LOADING SENTENCE TRANSFORMER MODEL")
print(f"{'='*60}")

if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
    model_path = CONFIG['paths']['pretrained_model_path']
    if Path(model_path).exists():
        st_model = SentenceTransformer(model_path)
        print(f"✓ Loaded pretrained model from: {model_path}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"⚠ Pretrained path not found, using base model")
else:
    st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
    print(f"✓ Loaded base model: {CONFIG['model']['base_model_name']}")

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=True
    )

print(f"\n✓ Model ready")
print(f"  Max sequence length: {st_model.max_seq_length}")
print(f"  Embedding dimension: {st_model.get_sentence_embedding_dimension()}")

In [ ]:
# ============================================================
# CELL 3.2: ENCODE VOCABULARY & BUILD NN INDEX
# ============================================================

print(f"\n{'='*60}")
print("ENCODING VOCABULARY")
print(f"{'='*60}")

vocab_emb = st_embed(terms)
print(f"✓ Encoded {len(terms)} terms → shape {vocab_emb.shape}")

term2idx = {t: i for i, t in enumerate(terms)}

print("\nBuilding nearest neighbor index...")
nn = NearestNeighbors(n_neighbors=min(100, len(terms)), metric="cosine", algorithm="auto")
nn.fit(vocab_emb)
print("✓ NN index ready")

def nearest_terms(query: str, k: int = 50):
    qv = st_embed([query])[0].reshape(1, -1)
    distances, indices = nn.kneighbors(qv, n_neighbors=min(k, len(terms)))
    sims = 1.0 - distances[0]
    idxs = indices[0]
    return [(terms[i], float(sims[j])) for j, i in enumerate(idxs)]

In [ ]:
# ============================================================
# CELL 3.3: EXPAND SEED TERMS
# ============================================================

print(f"\n{'='*60}")
print("EXPANDING SEED TERMS")
print(f"{'='*60}")

topic_rows = []
k = CONFIG['expand']['k_nearest']
min_cos = CONFIG['expand']['min_cosine']

for topic, seeds in CONFIG['topics'].items():
    print(f"  Processing: {topic}")
    seen = {}
    for s in seeds:
        for w, sim in nearest_terms(s, k=k):
            if doc_freq.get(w, 0) < CONFIG['vocab']['min_df']:
                continue
            if sim < min_cos:
                continue
            if w not in seen or sim > seen[w]:
                seen[w] = sim
    
    for s in seeds:
        if s in term2idx:
            seen[s] = max(seen.get(s, 0.0), 1.0)
    
    rows = sorted(seen.items(), key=lambda x: x[1], reverse=True)[:CONFIG['expand']['topN_per_topic']]
    for w, sc in rows:
        topic_rows.append({
            "topic": topic,
            "term": w,
            "cosine": round(sc, 4),
            "df": int(doc_freq.get(w, 0))
        })
    print(f"    → Found {len(rows)} candidate terms")

cands_df = pd.DataFrame(topic_rows)
print(f"\n✓ Generated {len(cands_df)} candidates across {cands_df['topic'].nunique()} topics")

fs.save_data(cands_df, "expanded_candidates", "Dictionary", "csv")

print(f"\n⚠️ MANUAL CURATION REQUIRED:")
print(f"  1. Review: Dictionary/expanded_candidates.csv")
print(f"  2. Remove irrelevant terms")
print(f"  3. Save as: Dictionary/curated_dictionary.csv")
print(f"  4. Then proceed to CHECKPOINT 4")

fs.save_config("checkpoint3_expansion")

✅ **CHECKPOINT 3 COMPLETE** - Dictionary expanded

⚠️ **STOP HERE** - Manually curate `expanded_candidates.csv` and save as `curated_dictionary.csv`

---
# CHECKPOINT 4: Topic Vector Creation
---

Build weighted topic vectors from curated dictionary.

In [ ]:
# ============================================================
# CELL 4.1: LOAD CURATED DICTIONARY & BUILD TOPIC VECTORS
# ============================================================

print(f"\n{'='*60}")
print("BUILDING TOPIC VECTORS FROM CURATED DICTIONARY")
print(f"{'='*60}")

curated_path = fs.folders['Dictionary'] / 'curated_dictionary.csv'

if not curated_path.exists():
    print(f"❌ Curated dictionary not found: {curated_path}")
    print(f"   Please complete manual curation first!")
else:
    pruned = pd.read_csv(curated_path)
    print(f"✓ Loaded curated dictionary: {len(pruned)} terms, {pruned['topic'].nunique()} topics")
    
    # Calculate SIF weights
    total_tf = max(1, sum(term_freq.values()))
    a = CONFIG['scoring']['sif_a']
    
    def term_weight(t: str) -> float:
        if not CONFIG['scoring']['use_sif']:
            return 1.0
        tf = term_freq.get(t, 1)
        return 1.0 / (a + tf / total_tf)
    
    # Build topic vectors
    topic2vec = {}
    topic2terms = defaultdict(list)
    
    for topic, sub in pruned.groupby('topic', sort=False):
        vecs, ws = [], []
        for t in sub['term']:
            if t not in term2idx:
                continue
            v = vocab_emb[term2idx[t]]
            w = term_weight(t)
            vecs.append(v)
            ws.append(w)
            topic2terms[topic].append(t)
        
        if not vecs:
            continue
        
        V = np.vstack(vecs)
        W = np.array(ws).reshape(-1, 1)
        tv = (V * W).sum(axis=0) / (W.sum() + 1e-12)
        tv = tv / (np.linalg.norm(tv) + 1e-12)
        topic2vec[topic] = tv
        print(f"  {topic}: {len(topic2terms[topic])} terms")
    
    print(f"\n✓ Created {len(topic2vec)} topic vectors")
    
    # Save
    fs.save_data(topic2vec, "topic_vectors", "Other_data", "npy")
    
    meta = {
        "topics": list(topic2vec.keys()),
        "terms": dict(topic2terms)
    }
    fs.save_data(meta, "topic_vectors_meta", "Other_data", "json")
    
    # Save per-topic suggestions
    for topic, sub in pruned.groupby('topic', sort=False):
        out = sub.copy()
        if 'topic' not in out.columns:
            out.insert(0, 'topic', topic)
        out['keep'] = True
        topic_filename = f"{topic.replace(' ', '_')}_suggestions.csv"
        topic_path = fs.folders['Dictionary_suggestions'] / topic_filename
        out.to_csv(topic_path, index=False, encoding='utf-8')
    
    print(f"✓ Saved per-topic suggestions to Dictionary_suggestions/")
    
    fs.save_config("checkpoint4_vectors")

✅ **CHECKPOINT 4 COMPLETE** - Topic vectors created

**Resume**: Load `topic2vec` from `Other_data/topic_vectors.npy`

---
# CHECKPOINT 5: Chunk Scoring & Confidence Classification
---

Score all chunks and classify by confidence level (High/Low/None).

In [ ]:
# ============================================================
# CELL 5.1: SCORE CHUNKS
# ============================================================

print(f"\n{'='*60}")
print("SCORING CHUNKS")
print(f"{'='*60}")

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Score all chunks
records = []
for idx, chunk in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Scoring"):
    text = chunk['text_for_scoring']
    
    if text:
        dv = st_embed([text])[0]
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': text,
        }
        for topic, tv in topic2vec.items():
            row[f'cos_{topic}'] = cosine(dv, tv)
    else:
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': '',
        }
        for topic in topic2vec.keys():
            row[f'cos_{topic}'] = 0.0
    
    records.append(row)

all_scores_df = pd.DataFrame(records)
print(f"\n✓ Scored {len(all_scores_df)} chunks across {len(topic2vec)} topics")

# Calculate metrics
topic_cols = [col for col in all_scores_df.columns if col.startswith('cos_')]
all_scores_df['max_score'] = all_scores_df[topic_cols].max(axis=1)
all_scores_df['primary_topic'] = all_scores_df[topic_cols].idxmax(axis=1).str.replace('cos_', '')

topic_scores = all_scores_df[topic_cols].values
sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
all_scores_df['score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]

In [ ]:
# ============================================================
# CELL 5.2: CONFIDENCE CLASSIFICATION
# ============================================================

print(f"\n{'='*60}")
print("CONFIDENCE CLASSIFICATION")
print(f"{'='*60}")

# Thresholds
HIGH_SCORE = CONFIG['scoring']['high_confidence_score']
HIGH_MARGIN = CONFIG['scoring']['high_confidence_margin']
LOW_SCORE = CONFIG['scoring']['low_confidence_score']
LOW_MARGIN = CONFIG['scoring']['low_confidence_margin']

# Classify
high_mask = (
    (all_scores_df['max_score'] >= HIGH_SCORE) & 
    (all_scores_df['score_margin'] >= HIGH_MARGIN)
)

low_mask = (
    (all_scores_df['max_score'] >= LOW_SCORE) & 
    (all_scores_df['score_margin'] >= LOW_MARGIN) &
    ~high_mask
)

no_mask = ~(high_mask | low_mask)

high_df = all_scores_df[high_mask].copy()
low_df = all_scores_df[low_mask].copy()
no_df = all_scores_df[no_mask].copy()

high_df['confidence_level'] = 'high'
low_df['confidence_level'] = 'low'
no_df['confidence_level'] = 'none'

# Save
fs.save_data(high_df, "scores_high_confidence", "Cosine_labeling", "csv")
fs.save_data(low_df, "scores_low_confidence", "Cosine_labeling", "csv")
fs.save_data(no_df, "scores_no_confidence", "Cosine_labeling", "csv")

all_labeled = pd.concat([high_df, low_df, no_df], ignore_index=True)
fs.save_data(all_labeled, "scores_all_labeled", "Cosine_labeling", "csv")

# Report
total = len(all_scores_df)
print(f"\nTotal chunks: {total}")
print(f"\n1. HIGH CONFIDENCE: {len(high_df)} ({len(high_df)/total*100:.1f}%)")
print(f"   Mean score: {high_df['max_score'].mean():.3f}, Mean margin: {high_df['score_margin'].mean():.3f}")
print(f"\n2. LOW CONFIDENCE: {len(low_df)} ({len(low_df)/total*100:.1f}%)")
print(f"   Mean score: {low_df['max_score'].mean():.3f}, Mean margin: {low_df['score_margin'].mean():.3f}")
print(f"\n3. NO CONFIDENCE: {len(no_df)} ({len(no_df)/total*100:.1f}%)")
print(f"   Mean score: {no_df['max_score'].mean():.3f}, Mean margin: {no_df['score_margin'].mean():.3f}")

fs.save_config("checkpoint5_scoring")

✅ **CHECKPOINT 5 COMPLETE** - Chunks scored and classified

**Resume**: Load confidence CSVs from `Cosine_labeling/`

---
# CHECKPOINT 6: Training Data Preparation
---

Create train/val splits from confidence tiers.

In [ ]:
# ============================================================
# CELL 6.1: PREPARE LABELED DATA
# ============================================================

print(f"\n{'='*60}")
print("PREPARING TRAINING DATA")
print(f"{'='*60}")

# High confidence = labeled training data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

# Split train/val
topic_counts = df_labeled['label'].value_counts()
can_stratify = all(topic_counts >= 2)

if can_stratify:
    train_df, val_df = train_test_split(
        df_labeled[['text', 'label', 'label_id']],
        test_size=0.2,
        stratify=df_labeled['label'],
        random_state=42
    )
    print(f"\n✓ Stratified split")
else:
    train_df, val_df = train_test_split(
        df_labeled[['text', 'label', 'label_id']],
        test_size=0.2,
        random_state=42
    )
    print(f"\n⚠ Random split (some topics < 2 examples)")

print(f"  Training: {len(train_df)}, Validation: {len(val_df)}")
train_df['is_pseudo'] = False
val_df['is_pseudo'] = False

In [ ]:
# ============================================================
# CELL 6.2: PREPARE UNLABELED & PSEUDO-LABELED DATA
# ============================================================

# Unlabeled (no confidence)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()
df_unlabeled['is_pseudo'] = False

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Pseudo-labeled (low confidence)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"Pseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample for balance
max_unlabeled = len(train_df) * 3
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
else:
    df_unlabeled_sampled = df_unlabeled

max_pseudo = len(train_df) * 2
if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo.sample(n=max_pseudo, random_state=42)
else:
    df_pseudo_sampled = df_pseudo

print(f"\nSampled for training:")
print(f"  Unlabeled: {len(df_unlabeled_sampled)}")
print(f"  Pseudo: {len(df_pseudo_sampled)}")

In [ ]:
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS
# ============================================================

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS")
print(f"{'='*60}")

# Option 1: High confidence only
train_opt1 = train_df.copy()

# Option 2: High + Unlabeled
train_opt2 = pd.concat([train_df, df_unlabeled_sampled], ignore_index=True)

# Option 3: High + Pseudo
train_opt3 = pd.concat([
    train_df, 
    df_pseudo_sampled[['text', 'label', 'label_id', 'is_pseudo']]
], ignore_index=True)

# Option 4: All three (RECOMMENDED)
train_opt4 = pd.concat([
    train_df,
    df_pseudo_sampled[['text', 'label', 'label_id', 'is_pseudo']],
    df_unlabeled_sampled
], ignore_index=True)

print(f"\n1. Option 1 (High only): {len(train_opt1)} examples")
print(f"2. Option 2 (High + Unlabeled): {len(train_opt2)} examples")
print(f"3. Option 3 (High + Pseudo): {len(train_opt3)} examples")
print(f"4. Option 4 (All) ⭐ RECOMMENDED: {len(train_opt4)} examples")

# Save all options
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
fs.save_data(val_df, "val_data", "Model_finetuning", "csv")
fs.save_data(train_opt1, "train_data_option1", "Model_finetuning", "csv")
fs.save_data(train_opt2, "train_data_option2", "Model_finetuning", "csv")
fs.save_data(train_opt3, "train_data_option3", "Model_finetuning", "csv")
fs.save_data(train_opt4, "train_data_option4", "Model_finetuning", "csv")

print(f"\n✓ All dataset options saved")

fs.save_config("checkpoint6_training_prep")

✅ **CHECKPOINT 6 COMPLETE** - Training data prepared

**Resume**: Load train/val CSVs from `Model_finetuning/`

---
# CHECKPOINT 7: Model Training (BERTJE)
---

Fine-tune Dutch BERT on labeled data.

⚠️ **Note**: This requires `transformers` library and GPU for efficient training.

In [ ]:
# ============================================================
# CELL 7.1: SETUP TRAINING
# ============================================================

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding
    )
    from datasets import Dataset
    import torch
    
    print("✓ Transformers library available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print("⚠ Transformers library not available")
    print("  Install: pip install transformers datasets torch")
    TRAINING_AVAILABLE = False

In [ ]:
# ============================================================
# CELL 7.2: LOAD & PREPARE DATA
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING TRAINING DATA")
    print(f"{'='*60}")
    
    # Select dataset option
    dataset_option = CONFIG['training']['dataset_option']
    
    if dataset_option == 'option1':
        train_dataset = train_opt1
    elif dataset_option == 'option2':
        train_dataset = train_opt2
    elif dataset_option == 'option3':
        train_dataset = train_opt3
    else:
        train_dataset = train_opt4
    
    print(f"\nUsing {dataset_option}: {len(train_dataset)} examples")
    
    # Load model & tokenizer
    model_name = "GroNLP/bert-base-dutch-cased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id
    )
    model.to(device)
    
    print(f"\n✓ Model loaded: {model_name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding=False,
            truncation=True,
            max_length=512
        )
    
    # Prepare labeled data
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy()
    
    hf_train = Dataset.from_pandas(train_labeled[['text', 'label_id']].reset_index(drop=True))
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_train = hf_train.rename_column('label_id', 'labels')
    hf_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    hf_val = Dataset.from_pandas(val_df[['text', 'label_id']].reset_index(drop=True))
    hf_val = hf_val.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val = hf_val.rename_column('label_id', 'labels')
    hf_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    print(f"\n✓ Data prepared")
    print(f"  Train: {len(hf_train)}, Val: {len(hf_val)}")

In [ ]:
# ============================================================
# CELL 7.3: TRAIN MODEL
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL")
    print(f"{'='*60}")
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    # Training arguments
    model_output_dir = str(fs.folders['Model_finetuning'])
    
    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    
    # Save model
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)
    
    # Evaluate
    eval_results = trainer.evaluate()
    
    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
    
    # Save metrics
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")
    
    print(f"\n✓ Model and metrics saved")
else:
    print("⚠ Skipping training - transformers library not available")

✅ **CHECKPOINT 7 COMPLETE** - Model trained and saved

**Resume**: Load model from `Model_finetuning/`

---
# CHECKPOINT 8: Visualizations
---

Generate interactive visualizations for analysis.

In [ ]:
# ============================================================
# CELL 8.1: SETUP VISUALIZATION LIBRARIES
# ============================================================

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans
    
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 8)
    
    print("✓ Visualization libraries loaded")
    VIZ_AVAILABLE = True
except ImportError:
    print("⚠ Visualization libraries not available")
    print("  Install: pip install matplotlib seaborn plotly scikit-learn")
    VIZ_AVAILABLE = False

In [ ]:
# ============================================================
# CELL 8.2: TOPIC DISTRIBUTION VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*60}")
    print("CREATING TOPIC DISTRIBUTION PLOT")
    print(f"{'='*60}")
    
    # Create distribution plots by confidence level
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['High Confidence', 'Low Confidence', 'No Confidence']
    )
    
    for idx, (df_conf, name) in enumerate([
        (high_df, 'High'),
        (low_df, 'Low'),
        (no_df, 'None')
    ], 1):
        topic_counts = df_conf['primary_topic'].value_counts()
        fig.add_trace(
            go.Bar(
                x=topic_counts.index,
                y=topic_counts.values,
                name=name
            ),
            row=1, col=idx
        )
    
    fig.update_layout(
        title="Topic Distribution by Confidence Level",
        height=500,
        showlegend=False
    )
    
    output_path = fs.folders['Visuals'] / 'topic_distribution.html'
    fig.write_html(str(output_path))
    print(f"✓ Saved: Visuals/topic_distribution.html")
else:
    print("⚠ Skipping visualizations")

In [ ]:
# ============================================================
# CELL 8.3: CONFIDENCE SCORE DISTRIBUTION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\nCreating confidence score distribution...")
    
    fig = go.Figure()
    
    for df_conf, name, color in [
        (high_df, 'High', 'green'),
        (low_df, 'Low', 'orange'),
        (no_df, 'None', 'red')
    ]:
        fig.add_trace(go.Histogram(
            x=df_conf['max_score'],
            name=name,
            opacity=0.7,
            marker_color=color
        ))
    
    fig.update_layout(
        title="Score Distribution by Confidence Level",
        xaxis_title="Max Cosine Score",
        yaxis_title="Count",
        barmode='overlay',
        height=500
    )
    
    output_path = fs.folders['Visuals'] / 'confidence_analysis.html'
    fig.write_html(str(output_path))
    print(f"✓ Saved: Visuals/confidence_analysis.html")

In [ ]:
# ============================================================
# CELL 8.4: TRAINING METRICS VISUALIZATION (if available)
# ============================================================

if VIZ_AVAILABLE:
    metrics_path = fs.folders['Model_finetuning'] / 'training_metrics.json'
    
    if metrics_path.exists():
        print(f"\nCreating training metrics visualization...")
        
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        fig = go.Figure()
        
        metric_names = ['accuracy', 'precision', 'recall', 'f1']
        metric_values = [
            metrics.get(f'eval_{m}', 0) for m in metric_names
        ]
        
        fig.add_trace(go.Bar(
            x=metric_names,
            y=metric_values,
            marker_color=['blue', 'green', 'orange', 'red']
        ))
        
        fig.update_layout(
            title=f"Model Performance Metrics (Dataset: {metrics.get('dataset_used', 'N/A')})",
            yaxis_title="Score",
            yaxis_range=[0, 1],
            height=500
        )
        
        output_path = fs.folders['Visuals'] / 'training_metrics.html'
        fig.write_html(str(output_path))
        print(f"✓ Saved: Visuals/training_metrics.html")
    else:
        print(f"\n  No training metrics found (model not trained yet)")
    
    fs.save_config("checkpoint8_visuals")
    
    print(f"\n{'='*60}")
    print("ALL VISUALIZATIONS COMPLETE")
    print(f"{'='*60}")
    print(f"\nSaved to: {fs.folders['Visuals']}")

✅ **CHECKPOINT 8 COMPLETE** - Visualizations generated

**All checkpoints complete!** Check `Visuals/` folder for interactive plots.

---
# Workflow Complete! 🎉
---

## Summary

All checkpoints have been executed:

✅ **CHECKPOINT 0**: Setup & Configuration
✅ **CHECKPOINT 1**: Text Processing
✅ **CHECKPOINT 2**: Vocabulary Building
✅ **CHECKPOINT 3**: Dictionary Expansion
✅ **CHECKPOINT 4**: Topic Vectors
✅ **CHECKPOINT 5**: Chunk Scoring
✅ **CHECKPOINT 6**: Training Data Prep
✅ **CHECKPOINT 7**: Model Training
✅ **CHECKPOINT 8**: Visualizations

## Output Location

All outputs saved to: `{workflow_root}`

```
{ModelType}-{Topic}_{Date}_{Version}/
├── config/              # Config snapshots at each checkpoint
├── Dictionary/          # Input, expanded, curated dictionaries
│   └── Dictionary_suggestions/
├── Model_finetuning/    # Trained model + metrics
├── Cosine_labeling/     # Confidence-classified scores
├── Bertje_labeling/     # Model predictions
├── Visuals/             # Interactive HTML visualizations
└── Other_data/          # Chunks, vocabulary, topic vectors
```

## Next Steps

1. **Review Results**: Check visualizations in `Visuals/`
2. **Analyze Model**: Review training metrics
3. **Use Model**: Load trained model for predictions
4. **Iterate**: Adjust config and re-run from any checkpoint

## Using the Trained Model

To use this model in a new workflow:

```python
CONFIG['model']['use_pretrained'] = True
CONFIG['paths']['pretrained_model_path'] = 'path/to/Model_finetuning'
CONFIG['workflow']['model_type'] = 'Finetuned_{Source}'
```

See `WORKFLOW_GUIDE_v3.md` for complete documentation!